# Подготовка Region из OpenStreetMap

Граница загружается через Nominatim, населённые пункты и дороги — через **Overpass**.
Загрузка, повторы, кэш и расчёты находятся в `popframe.preprocessing.data_loader`.
Здесь остаются только основные шаги и проверки. Итог — **один pickle модели Region**.

Запускайте ноутбук из папки `examples` с ядром `.venv`. После обновления библиотеки
перезапустите ядро. Поиск корня проекта, PBF-файлы и дополнительные зависимости не нужны.

## 1. Регион

Достаточно указать регион. Загрузчик использует служебный кэш `cache/osm`:
готовые ответы и участки не скачиваются заново. Это кэш для продолжения загрузки,
а не отдельные итоговые наборы данных. Для другого региона выбирайте другую папку
через `cache_dir`; несовместимые параметры кэша вызывают ошибку.

In [ ]:
from IPython.display import display

from popframe.preprocessing.data_loader import RegionDataLoader, build_region, save_region

loader = RegionDataLoader({"state": "Ленинградская область", "country": "Россия"})

## 2. Граница региона

Загрузчик проверяет геометрию, исправляет некорректные полигоны и создаёт рабочую копию
в локальной UTM с метрами. Исходный pickle для сборки не требуется и не изменяется.

In [ ]:
boundary = loader.load_boundary()
print("Рабочая проекция:", boundary.crs)

## 3. Населённые пункты и население

Загружаются именованные узлы `place=city/town/village/hamlet/isolated_dwelling`.
Полигоны place не дублируют центры. ID — это ID узлов OSM; они могут отличаться
от ID старой модели. Каждый участок сохраняется в кэш отдельно; после сбоя
нужно повторить только вызов `load_towns()`. На временные ошибки предусмотрено
до четырёх попыток с паузами, HTTP-код и причина выводятся в сообщении.

Население в OSM бывает неполным и устаревшим. Пропуски не заменяются нулями и пункты
не исключаются. Дополните `towns.population` по ID из проверенного источника перед
последней ячейкой. Для существующего CSV со столбцами `id,population,source` можно
использовать функцию `apply_population_updates` из того же модуля. CSV автоматически
не создаётся. Для исправления населения дорожный граф повторно загружать не нужно.

In [ ]:
towns = loader.load_towns()
display(towns[["id", "name", "population"]].head(10))
print("Без населения:", towns.population.isna().sum())
display(towns.loc[towns.population.isna(), ["id", "name"]].head(20))

## 4. Дорожный граф

Используется направленная автомобильная сеть. Внутренние отверстия границы заполняются
и добавляется буфер 20 км, рассчитанный в метрической проекции. Это позволяет маршрутам
проходить через Санкт-Петербург. Пункты остаются внутри исходной границы области.
При необходимости буфер задаётся через `routing_buffer_m` при создании загрузчика.

Скорость берётся из `maxspeed`, при его отсутствии оценивается по классу дороги.
Пробки, поворотные ограничения и пограничные проверки не моделируются. Недоступность
публичного сервера не устраняется повторами гарантированно: при 500/502/504 или таймауте
готовые участки сохраняются, а сообщение указывает причину и папку кэша.

In [ ]:
graph = loader.load_graph()
print("Дорожных узлов:", graph.number_of_nodes(), "Рёбер:", graph.number_of_edges())

## 5. Собрать модель и сохранить pickle

`build_region` проверяет население и геометрии, назначает категории через `LevelFiller`,
привязывает пункты к дорогам и вычисляет направленную матрицу **в минутах**.
ID строк и столбцов совпадают с ID пунктов. Недостижимые пары остаются `np.inf`.
Подъезд к ближайшему дорожному узлу не включается во время поездки; пункты на одном
узле могут иметь нулевое время между собой. Привязка дальше 2 км блокирует сборку.

При пропусках населения или ошибках данных будет ошибка с ID для проверки.
После их исправления достаточно повторить последнюю ячейку. Итоговый pickle появляется
только после успешной записи. Существующий файл не перезаписывается — при необходимости
укажите другое имя. Отдельных итоговых GeoJSON, Parquet, GraphML или CSV нет.

In [ ]:
region_model = build_region(boundary, towns, graph)
save_region(region_model, "data/Ленинградская область_osm.pickle")

Готовую модель далее можно открыть обычным `Region.from_pickle(...)`.
Источники: [OpenStreetMap и ODbL](https://www.openstreetmap.org/copyright),
[Overpass API](https://wiki.openstreetmap.org/wiki/Overpass_API).
Локальные проверки находятся в `tests/test_data_loader.py` и `tests/test_region_from_osm_notebook.py`.
Полная сетевая загрузка области и расчёт большого графа не проверялись до конца.